In [1]:
import numpy as np
import pandas as pd
import pyedflib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dropout, Flatten, Dense, Conv1D, MaxPooling1D

In [2]:
dataset = pd.read_csv('to_train_data.csv')

In [5]:
data = dataset[['FrL', 'FrR', 'OcR', 'target', 'FrL_RSI_100', 'FrR_RSI_100', 'OcR_RSI_100', 'FrL_DPO_100', 'FrR_DPO_100', 'OcR_DPO_100', 'FrL_momentum_100', 'FrR_momentum_100', 'OcR_momentum_100', 'FrL_stachostic_100', 'FrR_stachostic_100', 'OcR_stachostic_100']]

In [8]:
data.shape

(156000, 16)

In [13]:
window_size = 400

num_samples = (len(data) // window_size) * window_size
data = data.iloc[:num_samples]  # Обрезаем лишние строки

X = data.drop(columns=['target']).values.reshape(-num_samples, window_size, 15)  # Преобразуем в форму (Batch Size, 800, 3)
y = data['target'].values[:num_samples:window_size]  # Каждое значение 'target' соответствует одному окну из 800

# Проверим итоговые формы X и y
print("Форма X:", X.shape)  # Должно быть (Количество выборок, 800, 3)
print("Форма y:", y.shape)  # Должно быть (Количество выборок,)

# Разделение на обучающую и валидационную выборки
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Форма X: (390, 400, 15)
Форма y: (390,)


In [15]:
model = Sequential([
    Input(shape=(400, 15)),  # Входной размер (window_size, num_channels)
    
    # CNN блок
    Conv1D(filters=32, kernel_size=50, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    Conv1D(filters=64, kernel_size=50, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    
    # LSTM слой
    LSTM(128, return_sequences=False),
    
    # Полносвязный блок
    Dropout(0.4),
    Flatten(),
    Dense(3, activation='softmax')  # 3 класса для классификации
])

# Компиляция модели
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
# Обучение модели
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))

# Оценка модели
model.evaluate(X_val, y_val)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)                    │ (None, 351, 32)             │          24,032 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_2 (MaxPooling1D)       │ (None, 175, 32)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 175, 32)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_3 (Conv1D)                    │ (None, 126, 64)             │         102,464 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_3 (MaxPooling1D)       │ (None, 63, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 63, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 128)                 │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 225,699 (881.64 KB)

 Trainable params: 225,699 (881.64 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - accuracy: 0.3599 - loss: 1.2631 - val_accuracy: 0.4615 - val_loss: 1.1035
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3658 - loss: 1.2838 - val_accuracy: 0.4615 - val_loss: 1.0387
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3667 - loss: 1.1955 - val_accuracy: 0.3718 - val_loss: 1.0698
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3329 - loss: 1.2217 - val_accuracy: 0.4615 - val_loss: 1.0456
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3972 - loss: 1.1590 - val_accuracy: 0.4615 - val_loss: 1.0295
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4131 - loss: 1.1216 - val_accuracy: 0.4615 - val_loss: 1.0415
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4493 - loss: 1.1016 - val_accuracy: 0.4615 - val_loss: 1.0692
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3514 - loss: 1.2438 - val_accuracy: 0.4615 - v

[1.049971103668213, 0.4615384638309479]